# Домашнє завдання №9 — Агент технічної підтримки (LangGraph v1)

**Мета:** побудувати агента, який послідовно виконує 4 вузли:
`classify_query -> search_faq -> draft_response -> check_escalation`.

**Важливо:** у цьому рішенні використано **LangGraph v1** і строгий послідовний граф без умовної маршрутизації.

## Як користуватися ноутбуком

Це **навчальний агент технічної підтримки на LangGraph v1**. Нижче перед кожною кодовою коміркою є коротке пояснення: **мета → вхідні дані → логіка → результат**. Код залишено без змін від виправленої версії домашнього завдання.

**Порядок роботи:**

1. Відкрий ноутбук у Google Colab та виконуй комірки **зверху вниз**.
2. Збережи `OPENAI_API_KEY` у **Colab Secrets** або введи його через прихований запит `getpass`.
3. Прочитай пояснення до кожного вузла, після чого виконай відповідну кодову комірку.
4. **Остання комірка** викликає `test_agent()` — саме вона запускає п'ять тестів та створює платні API-запити.
5. Переглянь тести й метрики. Результати LLM можуть трохи відрізнятися між запусками.

**Обмеження прикладу:** пошук у FAQ працює через категорії та LLM, а не через векторну базу; ескалація лише позначається логічним прапорцем, без реальної інтеграції зі службою підтримки. `InMemorySaver` не зберігає checkpoint після завершення процесу.

**Ключові поняття:** `State` — спільні дані між вузлами; `Node` — функція обробки; `Edge` — перехід; `LLM-as-a-Judge` — окрема модельна оцінка відповіді.

## 1. Встановлення залежностей

**Мета комірки:** підготувати Python-середовище для запуску агента.

- `subprocess.check_call(...)` запускає команду встановлення бібліотек через `pip` у поточному Python-середовищі (`sys.executable`).
- `langgraph>=1.0` потрібний для побудови графа; `langchain-openai` — для звернень до моделей OpenAI; `langchain-core` — для базових інтерфейсів LangChain.
- Параметр `-U` дозволяє оновлювати пакети. Якщо команда успішна, `check_call` повертає **`0`**. Ненульовий код завершення викликає помилку.

**Що отримує:** доступ до `pip` та інтернету. **Що повертає:** встановлені пакети; не виконує запитів до OpenAI і не витрачає API-токени.

> У навчальному ноутбуці використано незакріплені версії деяких пакетів. Для довгострокової відтворюваності результати варто перевіряти з конкретними версіями.

In [ ]:
# Встановлення залежностей (LangGraph v1)
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "langgraph>=1.0",
        "langchain-openai",
        "langchain-core",
    ]
)

## 2. Безпечне завантаження OpenAI API-ключа

**Мета комірки:** надати бібліотеці OpenAI ключ доступу, не записуючи його безпосередньо в код.

Ключ перевіряється в такому порядку:

1. Змінна середовища `OPENAI_API_KEY`.
2. **Google Colab → Secrets** (`userdata.get("OPENAI_API_KEY")`).
3. Приховане введення через `getpass`, якщо попередні способи не спрацювали.

Якщо ключа немає, програма зупиниться з `ValueError`. Повідомлення `OPENAI_API_KEY: OK` означає лише, що непорожній ключ отримано, **а не те, що OpenAI вже підтвердив його чинність**.

**Що отримує:** секретний ключ. **Що повертає:** змінну середовища для наступних викликів `ChatOpenAI`.

> Не вставляй ключ у текст комірки, не додавай його до Git і не передавай разом із домашнім завданням.

In [ ]:
import os
from getpass import getpass

# Безпечне завантаження API ключа: env -> Colab Secrets -> ручне введення
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    try:
        from google.colab import userdata
        colab_key = userdata.get("OPENAI_API_KEY")
        if colab_key:
            api_key = colab_key
    except Exception:
        pass

if not api_key:
    api_key = getpass("Введіть OPENAI_API_KEY: ").strip()

if not api_key:
    raise ValueError("OPENAI_API_KEY не задано.")

os.environ["OPENAI_API_KEY"] = api_key
print("OPENAI_API_KEY: OK")

## 3. Імпорт інструментів і типів

**Мета комірки:** підключити всі класи й функції, необхідні для подальшого коду.

- `TypedDict`, `Literal`, `Optional`, `List` — для опису структури даних і підказок типів Python.
- `ChatOpenAI` — інтерфейс для виклику мовної моделі через OpenAI API.
- `StateGraph`, `START`, `END` — для побудови графа LangGraph та його початкової й кінцевої точок.
- `InMemorySaver` — для збереження стану графа **в оперативній пам'яті**.
- `BaseModel`, `Field` із Pydantic — для структурованих відповідей LLM та їх перевірки.

**Що отримує:** встановлені бібліотеки. **Що повертає:** доступні імена для наступних комірок; API-викликів тут немає.

In [ ]:
from typing import List, Literal, Optional, TypedDict

from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

## 4. Схеми даних, стан агента, FAQ та модель

**Мета комірки:** визначити, **які дані рухатимуться між вузлами**, де зберігаються відповіді довідника і яку модель використовує агент.

- `QueryClassification(TypedDict)` задає формат класифікації: `type` (тип звернення), `category` (категорія), `urgency` (терміновість), `summary` (короткий опис). `Literal` обмежує очікувані значення для типізації.
- `SupportAgentState(TypedDict)` описує спільний **state**: оригінальний запит, класифікацію, знайдені FAQ, чернетку та прапорець ескалації.
- `FAQ_DATABASE` — **демонстраційна база знань**: звичайний Python-словник із п'ятьма категоріями та готовими текстами.
- `llm = ChatOpenAI(...)` створює об'єкт для майбутніх звернень до моделі `gpt-4o-mini`. `temperature=0` зменшує випадковість, але не гарантує однакових результатів за кожного запуску.

**Що отримує:** нічого зовнішнього. **Що повертає:** схеми, словник FAQ й об'єкт клієнта моделі. **Ця комірка сама по собі не надсилає запит до OpenAI**.

> `TypedDict` описує очікувану структуру словника, але **не робить повної перевірки в runtime**. Поля типу `Optional[...]` можуть мати значення `None`; це не означає, що ключ автоматично стає необов'язковим.
>
> Записи FAQ — умови навчального завдання. Наприклад, повернення подвійного списання за 3–5 днів тут **не є перевіреною політикою реальної компанії**.

In [ ]:
# ТИПИ ДАНИХ (з умови)
class QueryClassification(TypedDict):
    """Структура класифікації"""
    type: Literal["problem", "question", "complaint"]
    category: Literal["password", "payment", "technical", "account", "general"]
    urgency: Literal["low", "medium", "high"]
    summary: str


class SupportAgentState(TypedDict):
    """Стан агента"""
    user_query: str
    classification: Optional[QueryClassification]
    search_results: Optional[List[str]]
    draft_response: Optional[str]
    needs_escalation: bool


FAQ_DATABASE = {
    "password": [
        "Скидання пароля: Натисніть 'Забули пароль?' -> введіть email -> перевірте пошту",
        "Новий пароль: мінімум 8 символів з літерами та цифрами",
        "Лист не прийшов - перевірте папку спам",
    ],
    "payment": [
        "Перевірте дані картки та баланс",
        "Подвійне списання повернеться автоматично за 3-5 днів",
        "Зміна тарифу: Профіль -> Підписка",
    ],
    "technical": [
        "Оновіть сторінку (F5) або очистьте кеш",
        "Спробуйте інший браузер",
        "Перевірте status.ourservice.com",
    ],
    "account": [
        "Перевірте правильність email",
        "Корпоративні акаунти: розділ 'Управління командою'",
        "Блокування - зверніться до підтримки",
    ],
    "general": [
        "Підтримка працює 24/7",
        "Час відповіді: 2-4 години",
    ],
}

# Дешева модель для виконання ДЗ
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 5. Структурована відповідь LLM і допоміжні функції

**Мета комірки:** зробити результат класифікації та вибір пунктів FAQ більш передбачуваними.

- `QueryClassificationSchema(BaseModel)` — Pydantic-схема відповіді класифікатора: LLM має повертати поля визначених типів і дозволених значень.
- `FAQSelection(BaseModel)` — схема вибору **номерів** релевантних пунктів довідника (список `indices`).
- `_limit_words(text, limit)` — обрізає текст до заданої кількості слів. Тут за слово умовно приймається фрагмент між пробілами (`split()`).
- `_safe_classification(data)` — переносить поля з об'єкта Pydantic у звичайний словник відповідно до `QueryClassification`.

**Важлива різниця:** `TypedDict` визначає типи для структури **стану графа**, тоді як Pydantic допомагає перевіряти **формат відповіді моделі**.

**Що отримує:** дані від LLM або текст. **Що повертає:** стандартизовані структури або обмежений за довжиною рядок. У цій комірці лише оголошуються класи й функції; їх ще не викликають.

In [ ]:
class QueryClassificationSchema(BaseModel):
    type: Literal["problem", "question", "complaint"]
    category: Literal["password", "payment", "technical", "account", "general"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="Коротке резюме запиту українською")


class FAQSelection(BaseModel):
    indices: List[int] = Field(
        description="Індекси 1-2 релевантних пунктів FAQ (нумерація з 1)"
    )


def _limit_words(text: str, limit: int) -> str:
    words = text.split()
    if len(words) <= limit:
        return text.strip()
    return " ".join(words[:limit]).strip()


def _safe_classification(data: QueryClassificationSchema) -> QueryClassification:
    return {
        "type": data.type,
        "category": data.category,
        "urgency": data.urgency,
        "summary": data.summary.strip(),
    }

## 6. Вузол `classify_query`: класифікація звернення

**Мета вузла:** з'ясувати, **який це запит**, **до якої категорії FAQ він належить** і **наскільки терміновий**.

**Вхід:** `state["user_query"]` — текст користувача.

**Що відбувається:**

1. Створюється промпт з правилами класифікації.
2. `llm.with_structured_output(QueryClassificationSchema)` налаштовує очікуваний формат відповіді.
3. `invoke(prompt)` **надсилає платний запит** до моделі.
4. `_safe_classification(...)` перетворює результат на словник.

**Вихід:** `{"classification": ...}` — **часткове оновлення стану**, а не новий стан цілком. Наприклад, `category="payment"` і `urgency="high"`.

**Наступний вузол:** завжди `search_faq`; рішення про ескалацію тут ще **не** змінює маршрут.

In [ ]:
def classify_query(state: SupportAgentState) -> dict:
    """Classify user intent/type, FAQ category, and urgency using OpenAI.

    Returns only the classification state update. The next node is always search_faq.
    """
    query = state["user_query"]

    prompt = f"""
Ти класифікатор запитів техпідтримки.
Поверни JSON за схемою.

Правила:
- type:
  - complaint: скарга, емоційне невдоволення, вимоги, звинувачення
  - problem: є проблема/збій у користувача без явної скарги
  - question: нейтральне інформаційне питання
- category:
  - password: пароль, вхід, скидання
  - payment: оплата, підписка, гроші, повернення, тариф, тарифний план
  - technical: сайт не працює, баг, помилка, збій
  - account: акаунт, профіль, корпоративний акаунт, користувачі команди
  - general: інше
- urgency:
  - high: шахрайство, подвійне списання, тривалий простій сервісу, негайні вимоги
  - medium: важлива, але не критична проблема доступу
  - low: звичайні інформаційні питання
- summary: 1 коротке речення українською.

Запит: {query}
""".strip()

    structured_llm = llm.with_structured_output(QueryClassificationSchema)
    classification_obj = structured_llm.invoke(prompt)
    classification = _safe_classification(classification_obj)

    return {"classification": classification}

## 7. Вузол `search_faq`: пошук відповідних пунктів довідника

**Мета вузла:** вибрати **1–2 найбільш релевантні пункти FAQ** для кожного запиту, включно з тими, що згодом будуть ескальовані.

**Вхід:** `user_query`, а також `classification.category` із попереднього вузла.

**Що відбувається:**

1. За категорією вибирається підсписок із `FAQ_DATABASE`.
2. Його пункти нумеруються, щоб модель могла назвати їхні індекси.
3. LLM повертає структурований список індексів (`FAQSelection`).
4. Python відкидає невалідні й дубльовані індекси та залишає максимум два пункти.

**Вихід:** `{"search_results": ["...", "..."]}`.

**Важливо:** це **не векторний retrieval**, не embeddings і не FAISS. Це пошук за категорією + **вибір FAQ через LLM**. Отже, тут є ще один платний запит. Якщо валідного результату немає, код явно повідомляє про помилку.

**Наступний вузол:** завжди `draft_response`.

In [ ]:
def search_faq(state: SupportAgentState) -> dict:
    """Select one or two category-specific FAQ entries for every query.

    The retrieval step is always executed, even for queries to be escalated.
    An API failure is deliberately raised, not silently replaced by an unrelated FAQ.
    """
    classification = state.get("classification") or {}
    category = classification.get("category", "general")
    if category not in FAQ_DATABASE:
        category = "general"

    faq_items = FAQ_DATABASE[category]
    numbered = "\n".join(f"{idx + 1}. {item}" for idx, item in enumerate(faq_items))

    prompt = f"""
Select 1 or 2 MOST relevant FAQ entries for the user query.
The indices are 1-based. Never use an index outside the list.
For a pricing-plan change question, prefer the plan-change FAQ,
not card or balance troubleshooting.
For a duplicate payment, prefer the duplicate-charge FAQ.
For a service outage, prefer the service-status FAQ.
For team invitations, prefer the team-management FAQ.

Category: {category}
User query: {state['user_query']}
FAQ entries:
{numbered}
""".strip()

    selector_llm = llm.with_structured_output(FAQSelection, method="json_schema")
    selection = selector_llm.invoke(prompt)
    valid_indices = []
    for idx in selection.indices:
        if isinstance(idx, int) and 1 <= idx <= len(faq_items) and idx not in valid_indices:
            valid_indices.append(idx)
    valid_indices = valid_indices[:2]
    if not valid_indices:
        raise ValueError(f"No valid FAQ indices for category {category!r}")
    return {"search_results": [faq_items[idx - 1] for idx in valid_indices]}

## 8. Вузол `draft_response`: генерація відповіді

**Мета вузла:** на основі запиту й знайдених FAQ сформувати **коротку конкретну відповідь українською**.

**Вхід:** `user_query`, `classification`, `search_results` зі стану.

**Що відбувається:**

1. Знайдені FAQ об'єднуються в текстовий контекст для промпту.
2. Модель отримує інструкції: не вигадувати факти, не стверджувати, що повернення коштів уже виконано, та не додавати повідомлення про ескалацію.
3. `llm.invoke(prompt)` генерує текст (**платний API-запит**).
4. `_limit_words(..., 40)` залишає до 40 слів, резервуючи місце для можливого повідомлення про ескалацію.

**Вихід:** `{"draft_response": "..."}`.

**Наступний вузол:** завжди `check_escalation`.

> Обмеження довжини не доводить, що відповідь фактично правильна: для цього є окремі перевірки якості.

In [ ]:
def draft_response(state: SupportAgentState) -> dict:
    """Draft a concise Ukrainian answer grounded in the selected FAQ items.

    Do not add the escalation notice here: check_escalation owns that step.
    Keep a 40-word budget for the answer; the final notice must fit in 50 words.
    """
    classification = state.get("classification") or {}
    search_results = state.get("search_results") or []
    faq_context = "\n".join(f"- {item}" for item in search_results)

    prompt = f"""
Respond to the user in Ukrainian only.
Rules:
- at most 40 words
- concrete steps, no filler or generic introduction
- use only facts present in the FAQ context
- do NOT claim a refund was initiated or a specialist was contacted
- do NOT add escalation notices; another node handles these
- when the user asks for a repair ETA but FAQ has none, do not invent one

User query: {state['user_query']}
Classification: {classification}
Relevant FAQ:
{faq_context}
""".strip()

    result = llm.invoke(prompt)
    response_text = result.content
    if not isinstance(response_text, str):
        raise TypeError("LLM returned non-text content for draft_response")
    response_text = _limit_words(response_text, 40)
    if not response_text:
        raise ValueError("LLM returned an empty draft_response")
    return {"draft_response": response_text}

## 9. Вузол `check_escalation`: визначення необхідності передачі людині

**Мета вузла:** виконати **правило з умови домашнього завдання**, не покладаючись на додаткове припущення LLM.

**Вхід:** `classification`, початковий `user_query` і `draft_response`.

**Умова ескалації (достатньо будь-якої однієї):**

- `urgency == "high"`;
- `type == "complaint"`;
- у запиті є фрагмент слова `шахрайств`, `судов` або `юридичн`.

**Вихід:** `{"needs_escalation": True/False, "draft_response": ...}`. Якщо потрібна ескалація, до відповіді додається одне коротке повідомлення. Фінальна довжина — **не більш як 50 слів**.

**Ключовий нюанс:** вузол **лише встановлює статус і формує повідомлення**. Він не створює справжнього ticket і не контактує зі спеціалістом. Логіка реалізована звичайними умовами Python — **без нового API-виклику**.

**Наступний крок:** завершення графа (`END`).

In [ ]:
ESCALATION_KEYWORDS = ("шахрайств", "судов", "юридичн")
ESCALATION_NOTICE = "⚠️ Запит потребує розгляду спеціаліста."

def check_escalation(state: SupportAgentState) -> dict:
    """Apply the exact escalation rule and append one notice, if needed.

    This sets a flag; it does NOT perform a real support escalation.
    """
    classification = state.get("classification") or {}
    query_lower = (state.get("user_query") or "").lower()

    needs_escalation = (
        classification.get("urgency") == "high"
        or classification.get("type") == "complaint"
        or any(keyword in query_lower for keyword in ESCALATION_KEYWORDS)
    )

    response = (state.get("draft_response") or "").strip()

    if needs_escalation:
        reserve_for_notice = len(ESCALATION_NOTICE.split())
        base_limit = max(1, 50 - reserve_for_notice)
        response = _limit_words(response, base_limit)
        response = f"{response}\n\n{ESCALATION_NOTICE}" if response else ESCALATION_NOTICE

    response = _limit_words(response, 50)

    return {
        "needs_escalation": needs_escalation,
        "draft_response": response,
    }

## 10. `build_support_agent()`: складання графа LangGraph

**Мета комірки:** з'єднати раніше визначені Python-функції в один робочий граф.

```text
START
  ↓
classify_query
  ↓
search_faq
  ↓
draft_response
  ↓
check_escalation
  ↓
END
```

- `StateGraph(SupportAgentState)` задає **схему стану**, який передаватиметься між вузлами.
- `add_node("name", function)` реєструє кожну Python-функцію як вузол.
- `add_edge(...)` створює **фіксовані** переходи — кожен запит проходить усі чотири вузли.
- `compile(checkpointer=InMemorySaver())` створює виконуваний граф із збереженням стану **лише в пам'яті поточного процесу**.
- `get_graph().draw_mermaid()` повертає текстове Mermaid-представлення структури.

**Що повертає:** скомпільований граф. Комірка **ще не обробляє запити** і не витрачає API-токени.

> Це домашнє завдання навмисно використовує **послідовний граф без умовних переходів**. Прапорець `needs_escalation` не змінює шлях виконання.

In [ ]:
def build_support_agent():
    """Compile the strictly sequential four-node LangGraph homework workflow."""
    workflow = StateGraph(SupportAgentState)

    workflow.add_node("classify_query", classify_query)
    workflow.add_node("search_faq", search_faq)
    workflow.add_node("draft_response", draft_response)
    workflow.add_node("check_escalation", check_escalation)

    workflow.add_edge(START, "classify_query")
    workflow.add_edge("classify_query", "search_faq")
    workflow.add_edge("search_faq", "draft_response")
    workflow.add_edge("draft_response", "check_escalation")
    workflow.add_edge("check_escalation", END)

    checkpointer = InMemorySaver()
    return workflow.compile(checkpointer=checkpointer)


agent = build_support_agent()
print(agent.get_graph().draw_mermaid())

## 11. Оцінювання та тестування агента

**Мета комірки:** визначити функції, які перевірятимуть готовий граф на **п'яти обов'язкових сценаріях** із домашнього завдання.

### `calculate_answer_relevancy()` — LLM-as-a-Judge

- Передає суддівській моделі **запит і відповідь** та просить оцінити, наскільки відповідь стосується запиту.
- `AnswerRelevancyAssessment` задає структуровані поля `score` (0–1) та `explanation`.
- **Це додатковий платний виклик LLM**. Оцінка є орієнтовною та **не гарантує** factual correctness або Faithfulness.

### `test_agent()` — тестовий прогін

- Послідовно передає п'ять заданих запитів у граф через `graph.invoke(...)`.
- Для кожного виводить **класифікацію**, **вибрані FAQ**, **готову відповідь** та **прапорець ескалації**.
- Перевіряє категорію проти очікуваної, приналежність 1–2 FAQ потрібній категорії, обмеження в 50 слів і правильність ескалації.
- Рахує **Task Completion Rate** (у цьому тесті: частка випадків із непорожньою відповіддю), **Answer Relevancy** і **Escalation Rate**. Очікувана частка ескалацій — **2/5 = 40%**.
- Якщо суддівський API-виклик завершиться помилкою, тест повідомить про неї окремо: це не означає, що вузли агента не працювали.

**Важливо:** звичайний успішний сценарій передбачає приблизно **4 виклики LLM на тест**: класифікація, вибір FAQ, генерація, оцінювання. Для 5 тестів це приблизно **20 платних викликів** (без повторних спроб). Ця комірка **лише визначає функції** — фактичні виклики почнуться в наступній.

In [ ]:
# LLM-as-a-Judge (separate from agent execution)
class AnswerRelevancyAssessment(BaseModel):
    """Structured subjective relevance grade from the evaluator model."""
    score: float = Field(description="Relevance score between 0 and 1")
    explanation: str = Field(description="One short reason for the score")


def calculate_answer_relevancy(query: str, response: str) -> float:
    """Evaluate whether the answer addresses the given user query.

    A judge score is NOT the same as factual accuracy or faithfulness.
    The scorer raises on API failure; we never silently invent a score.
    """
    prompt = f"""
Evaluate answer relevance to the user question, on a scale of 0 to 1.
1.0: directly answers every part of the question.
0.5: partially answers the question.
0.0: does not address the question.
Do not score factual correctness; score relevance only.

Question: {query}
Answer: {response}
""".strip()
    judge = llm.with_structured_output(AnswerRelevancyAssessment, method="json_schema")
    grade = judge.invoke(prompt)
    return max(0.0, min(1.0, float(grade.score)))


def test_agent():
    """Run five assignment scenarios and show expected-vs-actual checks.

    Each scenario traverses classify_query -> search_faq -> draft_response ->
    check_escalation. The model evaluator makes extra paid API calls.
    Errors from the evaluator do not change the actual task completion rate.
    """
    cases = [
    (
        "Я забув пароль від свого акаунту, як його відновити?",
        "password",
        False
    ),
    (
        "З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!",
        "payment",
        True
    ),
    (
        "Чи можу я змінити тарифний план на дешевший?",
        "payment",
        False
    ),
    (
        "Сайт не працює вже третю годину! Коли полагодите?",
        "technical",
        True
    ),
    (
        "Як додати нового користувача до корпоративного акаунту?",
        "account",
        False
    ),
]

    graph = build_support_agent()
    results = []
    judge_scores = []
    completed = 0
    escalated = 0
    format_passes = 0
    category_passes = 0
    escalation_passes = 0

    print("=" * 70)
    print("SUPPORT AGENT HOMEWORK: FIVE CASES")
    print("=" * 70)

    for i, (query, expected_category, expected_escalation) in enumerate(cases, start=1):
        print(f"\n--- Test {i}/5 ---")
        print("QUERY:", query)
        config = {"configurable": {"thread_id": f"support_hw9_case_{i}"}}
        try:
            result = graph.invoke(
                {"user_query": query, "needs_escalation": False},
                config=config,
            )
        except Exception as exc:
            print(f"AGENT ERROR ({type(exc).__name__}): {exc}")
            results.append({})
            continue

        results.append(result)
        cls = result.get("classification") or {}
        found = result.get("search_results") or []
        response = (result.get("draft_response") or "").strip()
        needs_escalation = bool(result.get("needs_escalation"))

        print("CLASSIFICATION:", cls)
        print("FAQ:")
        for item in found:
            print("  -", item)
        print("RESPONSE:", response)
        print("NEEDS HUMAN ESCALATION:", needs_escalation)

        category_ok = cls.get("category") == expected_category
        escalation_ok = needs_escalation == expected_escalation
        faq_ok = 1 <= len(found) <= 2 and all(item in FAQ_DATABASE.get(expected_category, []) for item in found)
        length_ok = 0 < len(response.split()) <= 50
        print("CHECKS: category=", category_ok, "FAQ=", faq_ok,
              "response <= 50 words=", length_ok, "escalation=", escalation_ok)

        completed += int(bool(response))
        escalated += int(needs_escalation)
        category_passes += int(category_ok)
        escalation_passes += int(escalation_ok)
        format_passes += int(length_ok and faq_ok)

        if response:
            try:
                score = calculate_answer_relevancy(query, response)
            except Exception as exc:
                print(f"JUDGE ERROR (not agent failure): {type(exc).__name__}: {exc}")
            else:
                judge_scores.append(score)
                print(f"Answer Relevancy (LLM-as-a-Judge): {score:.0%}")

    count = len(cases)
    print("\n" + "=" * 70)
    print(f"Task Completion Rate: {completed / count:.0%} ({completed}/{count})")
    if judge_scores:
        print(f"Average Answer Relevancy: {sum(judge_scores) / len(judge_scores):.0%} "
              f"({len(judge_scores)}/{count} judge scores)")
    else:
        print("Average Answer Relevancy: unavailable (evaluator did not run successfully)")
    print(f"Escalation Rate: {escalated / count:.0%} ({escalated}/{count}); expected: 40%")
    print(f"Category checks: {category_passes}/{count}")
    print(f"FAQ and answer-length checks: {format_passes}/{count}")
    print(f"Escalation checks: {escalation_passes}/{count}")
    print("=" * 70)
    return results

## 12. Запуск тестування

**Мета комірки:** викликати `test_agent()` і **справді виконати** всі п'ять сценаріїв.

- `results = test_agent()` запускає граф для кожного прикладу та зберігає результати у списку `results`.
- Після виконання переглянь **фактичні категорії**, **вибрані FAQ**, **довжину відповідей**, **статуси ескалації** та підсумкові метрики.
- Ціль за умовою: 5 відповідей із 5, середня **Answer Relevancy ≥ 0.8**, ескалація **2/5 (40%)**.
- Якщо щось не спрацює, перевір **повідомлення `AGENT ERROR` та `JUDGE ERROR`** — вони позначають різні причини.

> ⚠️ **Саме ця комірка виконує платні запити до OpenAI API.** Повторний запуск тестів знову витратить токени. Навіть якщо відповіді схожі на наведені в умові, їх потрібно перевіряти, а не вважати гарантованими.

In [ ]:
# Run all five tests (paid OpenAI API calls)
results = test_agent()